#  Gravitational Lens Classifier 

Binary classification of strong gravitational lenses vs non-lensed galaxies.
- Phase 1: Supervised training with Focal Loss + Weighted Sampling
- Phase 2: RL fine-tuning (REINFORCE) to directly optimize AUC

## 1. Imports & Configuration

In [1]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
from torchvision import models, transforms
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, f1_score,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "dataset" / "task2"
OUTPUT_DIR = Path.cwd()

TRAIN_LENSES    = DATA_DIR / "train_lenses"
TRAIN_NONLENSES = DATA_DIR / "train_nonlenses"
TEST_LENSES     = DATA_DIR / "test_lenses"
TEST_NONLENSES  = DATA_DIR / "test_nonlenses"

SEED            = 42
BATCH_SIZE      = 64
PHASE1_EPOCHS   = 30
PHASE1_LR       = 1e-3
PHASE2_EPOCHS   = 30
PHASE2_LR       = 3e-4
RL_LR           = 1e-3
NUM_WORKERS     = 4
VAL_SPLIT       = 0.15  # fraction of training data used for validation

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 2. Dataset & Transforms

In [3]:
class LensDataset(Dataset):
    """Dataset for loading 3×64×64 .npy gravitational lensing images."""

    def __init__(self, file_paths, labels, augment=False):
        self.file_paths = file_paths
        self.labels = labels
        self.augment = augment
        self.transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomApply([
                transforms.RandomRotation(degrees=90),
            ], p=0.5),
            transforms.RandomApply([
                transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            ], p=0.3),
        ]) if augment else None

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        image = np.load(self.file_paths[idx]).astype(np.float32)  # (3, 64, 64)
        tensor = torch.from_numpy(image)
        if self.augment and self.transform:
            tensor = self.transform(tensor)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return tensor, label


def load_file_list(lenses_dir, nonlenses_dir):
    """Build file path and label lists from lens/non-lens directories."""
    paths, labels = [], []
    for f in sorted(lenses_dir.glob("*.npy")):
        paths.append(str(f))
        labels.append(1)
    for f in sorted(nonlenses_dir.glob("*.npy")):
        paths.append(str(f))
        labels.append(0)
    return paths, labels


def make_weighted_sampler(labels):
    """Create a WeightedRandomSampler to balance classes in each mini-batch."""
    labels_arr = np.array(labels)
    class_counts = np.bincount(labels_arr)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels_arr]
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(labels),
        replacement=True,
    )
    return sampler


def create_dataloaders():
    """Create train, val, and test data loaders with stratified val split."""
    # Load training data
    train_paths, train_labels = load_file_list(TRAIN_LENSES, TRAIN_NONLENSES)

    # Stratified train/val split
    lens_indices = [i for i, l in enumerate(train_labels) if l == 1]
    nonlens_indices = [i for i, l in enumerate(train_labels) if l == 0]

    rng = np.random.RandomState(SEED)
    rng.shuffle(lens_indices)
    rng.shuffle(nonlens_indices)

    n_val_lens = max(1, int(len(lens_indices) * VAL_SPLIT))
    n_val_nonlens = max(1, int(len(nonlens_indices) * VAL_SPLIT))

    val_indices = lens_indices[:n_val_lens] + nonlens_indices[:n_val_nonlens]
    tr_indices  = lens_indices[n_val_lens:] + nonlens_indices[n_val_nonlens:]

    tr_paths  = [train_paths[i] for i in tr_indices]
    tr_labels = [train_labels[i] for i in tr_indices]
    val_paths  = [train_paths[i] for i in val_indices]
    val_labels = [train_labels[i] for i in val_indices]

    # Test data
    test_paths, test_labels = load_file_list(TEST_LENSES, TEST_NONLENSES)

    print(f"📊 Data Summary:")
    print(f"   Train: {len(tr_paths):,}  (lens: {sum(tr_labels):,}, non-lens: {len(tr_labels)-sum(tr_labels):,})")
    print(f"   Val:   {len(val_paths):,}  (lens: {sum(val_labels):,}, non-lens: {len(val_labels)-sum(val_labels):,})")
    print(f"   Test:  {len(test_paths):,}  (lens: {sum(test_labels):,}, non-lens: {len(test_labels)-sum(test_labels):,})")

    train_dataset = LensDataset(tr_paths, tr_labels, augment=True)
    val_dataset   = LensDataset(val_paths, val_labels, augment=False)
    test_dataset  = LensDataset(test_paths, test_labels, augment=False)

    sampler = make_weighted_sampler(tr_labels)

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )

    return train_loader, val_loader, test_loader

## 3. Model

In [4]:
class LensClassifier(nn.Module):
    """ResNet18-based binary classifier for 3×64×64 lens images."""

    def __init__(self, dropout=0.3):
        super().__init__()
        backbone = models.resnet18(weights=None)
        # Keep 3-channel input (matches our 3 SDSS filters)
        # Replace the final FC layer
        num_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.features = backbone
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_features, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        feat = self.features(x)
        logit = self.classifier(feat)
        return logit.squeeze(-1)  # (B,)

    def get_features(self, x):
        """Extract features before classifier head (for RL phase)."""
        return self.features(x)

# FOCAL LOSS

class FocalLoss(nn.Module):
    """Binary Focal Loss for handling class imbalance.
    
    FL(p) = -α (1-p)^γ log(p)  for positive class
    FL(p) = -(1-α) p^γ log(1-p)  for negative class
    """

    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        loss = focal_weight * bce
        return loss.mean()

## 4. Training

In [5]:
def evaluate(model, loader, criterion, device):
    """Evaluate model on a data loader, return loss, AUC, predictions."""
    model.eval()
    all_targets, all_probs = [], []
    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for images, targets in loader:
            images, targets = images.to(device), targets.to(device)
            with autocast("cuda"):
                logits = model(images)
                loss = criterion(logits, targets)
            total_loss += loss.item()
            n_batches += 1
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
            all_targets.extend(targets.cpu().numpy())

    all_targets = np.array(all_targets)
    all_probs = np.array(all_probs)
    avg_loss = total_loss / max(n_batches, 1)

    try:
        auc_score = roc_auc_score(all_targets, all_probs)
    except ValueError:
        auc_score = 0.0

    return avg_loss, auc_score, all_targets, all_probs

### 4.1 Phase-1 Training

In [6]:
def train_phase1(model, train_loader, val_loader, device):
    """Phase 1: Supervised training with Focal Loss."""
    print("\n" + "=" * 70)
    print("🚀 PHASE 1: Supervised Training (Focal Loss + Weighted Sampling)")
    print("=" * 70)

    criterion = FocalLoss(alpha=0.75, gamma=2.0)
    optimizer = optim.Adam(model.parameters(), lr=PHASE1_LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS, eta_min=1e-6)
    scaler = GradScaler("cuda")

    history = {
        "train_loss": [], "val_loss": [],
        "val_auc": [], "lr": [],
    }
    best_auc = 0.0
    patience, patience_counter = 8, 0
    best_model_path = OUTPUT_DIR / "best_lens_model_phase1.pth"

    for epoch in range(PHASE1_EPOCHS):
        model.train()
        running_loss = 0.0
        n_batches = 0

        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()

            with autocast("cuda"):
                logits = model(images)
                loss = criterion(logits, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            n_batches += 1

        scheduler.step()
        avg_train_loss = running_loss / max(n_batches, 1)

        # Validate
        val_loss, val_auc, _, _ = evaluate(model, val_loader, criterion, device)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(val_loss)
        history["val_auc"].append(val_auc)
        history["lr"].append(current_lr)

        marker = ""
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), best_model_path)
            marker = " ✅ saved"
            patience_counter = 0
        else:
            patience_counter += 1

        print(
            f"  Epoch [{epoch+1:02d}/{PHASE1_EPOCHS}] "
            f"| Train Loss: {avg_train_loss:.4f} "
            f"| Val Loss: {val_loss:.4f} "
            f"| Val AUC: {val_auc:.4f} "
            f"| LR: {current_lr:.6f}{marker}"
        )

        if patience_counter >= patience:
            print(f"  ⏹ Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
            break

    # Reload best model
    model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
    print(f"\n  🏆 Phase 1 Best Val AUC: {best_auc:.4f}")
    return model, history, best_auc

### 4.2 Phase-2 Training

## 2. The RL Algorithm: REINFORCE with Baseline

Since the reward metrics(AUC, F1) are non-differentiable. So I used a Monte Carlo policy gradient algorithm.

### The Policy Update Rule
The policy network's weights ($\theta$) are updated using the log-derivative trick:
$$\nabla_\theta J(\theta) \approx \nabla_\theta \log \pi_\theta(A_t | S_t) \times (R_t - b)$$

Where:
* $\pi_\theta(A_t | S_t)$ is the probability of the sampled threshold shift given the batch statistics.
* $R_t$ is the computed reward.
* $b$ is a **baseline**.

### Variance Reduction (The Moving Baseline)
To stabilize training, Exponential Moving Average (EMA) baseline is employed:

`baseline_reward = 0.95 * baseline_reward + 0.05 * reward`

By subtracting this baseline from the reward we get the 

`advantage = reward - baseline_reward`

This should speeds up convergence.

---

## 3. Reward-Modulated Supervised Learning

Instead of updating the base model using pure RL, I used a **Reward-Scaled Supervised Loss**:

```python
reward_scale = max(0.1, min(2.0, reward / max(baseline_reward, 0.1)))
scaled_loss = sup_loss * reward_scale
scaled_loss.backward()

In [35]:
class ThresholdPolicyNetwork(nn.Module):
    """Small policy network that learns optimal classification threshold.
    
    Takes batch-level statistics of the base model's predictions and outputs
    a threshold adjustment via a Gaussian policy.
    
    State: [mean_prob, std_prob, median_prob, frac_above_0.5, skewness]
    Action: threshold shift (continuous) sampled from N(μ, σ²)
    """

    def __init__(self, state_dim=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.mu_head = nn.Linear(32, 1)   # mean of threshold shift
        self.log_std = nn.Parameter(torch.tensor(-1.0))  # learnable log std

    def forward(self, state):
        h = self.net(state)
        mu = torch.tanh(self.mu_head(h)) * 0.4  # clip to ±0.4 shift
        std = torch.exp(self.log_std).clamp(min=1e-4, max=0.3)
        return mu.squeeze(-1), std


def compute_batch_state(probs):
    """Compute state features from a batch of predicted probabilities."""
    mean_p = probs.mean()
    std_p = probs.std() + 1e-8
    median_p = probs.median()
    frac_above = (probs > 0.5).float().mean()
    # Skewness
    skew = ((probs - mean_p) ** 3).mean() / (std_p ** 3 + 1e-8)
    return torch.stack([mean_p, std_p, median_p, frac_above, skew])


def compute_auc_reward(probs_np, targets_np, threshold):
    """Compute AUC-based reward for the RL agent."""
    try:
        auc_score = roc_auc_score(targets_np, probs_np)
    except ValueError:
        auc_score = 0.5

    # Also compute F1 at the given threshold for bonus reward
    preds = (probs_np >= threshold).astype(int)
    try:
        f1 = f1_score(targets_np, preds, zero_division=0)
    except Exception:
        f1 = 0.0

    # Combined reward: AUC (ranking quality) + F1 bonus (threshold quality)
    reward = 0.7 * auc_score + 0.3 * f1
    return reward, auc_score


def train_phase2_rl(model, train_loader, val_loader, device):
    """Phase 2: RL fine-tuning to optimize AUC via REINFORCE.
    
    We fine-tune the classifier head while the policy network learns
    to adjust thresholds. The base model's last layers are also updated
    with a small learning rate guided by the RL reward signal.
    """
    print("\n" + "=" * 70)
    print("🤖 PHASE 2: RL-Based AUC Optimization (REINFORCE)")
    print("=" * 70)

    # Freeze feature extractor, only fine-tune classifier head
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

    # Policy network
    policy = ThresholdPolicyNetwork(state_dim=5).to(device)
    policy_optimizer = optim.Adam(policy.parameters(), lr=RL_LR)

    # Model fine-tuning optimizer (small lr)
    model_optimizer = optim.AdamW(
        model.classifier.parameters(), lr=PHASE2_LR, weight_decay=1e-5
    )
    criterion = FocalLoss(alpha=0.75, gamma=2.0)

    baseline_reward = 0.5  # running mean baseline for variance reduction
    baseline_decay = 0.95
    best_val_auc = 0.0
    best_model_path = OUTPUT_DIR / "best_lens_model.pth"
    history_rl = {"val_auc": [], "reward": [], "threshold": []}

    for epoch in range(PHASE2_EPOCHS):
        model.train()
        epoch_rewards = []
        epoch_thresholds = []
        policy_losses = []

        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)

            # Forward pass through base model
            with autocast("cuda"):
                logits = model(images)
                sup_loss = criterion(logits, targets)

            probs = torch.sigmoid(logits.detach().float())

            # RL: compute state and sample action
            state = compute_batch_state(probs).unsqueeze(0)  # (1, 5)
            mu, std = policy(state)
            dist = torch.distributions.Normal(mu, std)
            action = dist.sample()  # threshold shift
            log_prob = dist.log_prob(action)

            threshold = (0.5 + action).clamp(0.05, 0.95).item()

            # Compute reward
            probs_np = probs.cpu().numpy()
            targets_np = targets.cpu().numpy()
            reward, batch_auc = compute_auc_reward(probs_np, targets_np, threshold)

            # REINFORCE loss
            advantage = reward - baseline_reward
            policy_loss = -log_prob * advantage
            policy_losses.append(policy_loss)

            # Update baseline
            baseline_reward = baseline_decay * baseline_reward + (1 - baseline_decay) * reward

            # Update model with combined supervised + RL-guided loss
            # The RL reward modulates the learning rate effectively:
            # when AUC is improving, we keep the gradient; otherwise we dampen it
            model_optimizer.zero_grad()
            reward_scale = max(0.1, min(2.0, reward / max(baseline_reward, 0.1)))
            scaled_loss = sup_loss * reward_scale
            scaled_loss.backward()
            model_optimizer.step()

            epoch_rewards.append(reward)
            epoch_thresholds.append(threshold)

        # Update policy network
        policy_optimizer.zero_grad()
        total_policy_loss = torch.stack(policy_losses).mean()
        total_policy_loss.backward()
        policy_optimizer.step()

        # Validate
        val_loss, val_auc, _, _ = evaluate(model, val_loader, criterion, device)
        avg_reward = np.mean(epoch_rewards)
        avg_threshold = np.mean(epoch_thresholds)

        history_rl["val_auc"].append(val_auc)
        history_rl["reward"].append(avg_reward)
        history_rl["threshold"].append(avg_threshold)

        marker = ""
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), best_model_path)
            marker = " ✅ saved"

        print(
            f"  Epoch [{epoch+1:02d}/{PHASE2_EPOCHS}] "
            f"| Val AUC: {val_auc:.4f} "
            f"| Avg Reward: {avg_reward:.4f} "
            f"| Avg Threshold: {avg_threshold:.3f}{marker}"
        )

    # Reload best model
    model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))

    # Unfreeze all layers for final state
    for param in model.parameters():
        param.requires_grad = True

    # Save policy network
    torch.save(policy.state_dict(), OUTPUT_DIR / "threshold_policy.pth")

    print(f"\n  🏆 Phase 2 Best Val AUC: {best_val_auc:.4f}")
    return model, policy, history_rl

## 5. Evalutation

In [28]:
def plot_sample_predictions(model, test_loader, device, threshold):
    """Plot a grid of sample predictions with confidence scores."""
    model.eval()

    # Gather some predictions
    all_images, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for images, targets in test_loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_images.append(images.cpu().numpy())
            all_targets.extend(targets.numpy())
            all_probs.extend(probs)
            if len(all_targets) >= 500:
                break

    all_images = np.concatenate(all_images, axis=0)
    all_targets = np.array(all_targets)
    all_probs = np.array(all_probs)

    # Select 4 true lenses and 4 true non-lenses
    lens_idx = np.where(all_targets == 1)[0]
    nonlens_idx = np.where(all_targets == 0)[0]

    n_show = min(4, len(lens_idx))
    chosen_lens = np.random.choice(lens_idx, n_show, replace=False) if len(lens_idx) >= n_show else lens_idx
    chosen_nonlens = np.random.choice(nonlens_idx, n_show, replace=False)
    chosen = np.concatenate([chosen_lens, chosen_nonlens])

    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
    if n_show == 1:
        axes = axes.reshape(2, 1)

    class_names = ["Non-Lens", "Lens"]

    for i, idx in enumerate(chosen):
        row = 0 if i < n_show else 1
        col = i if i < n_show else i - n_show
        ax = axes[row, col]

        # Create RGB composite from the 3 filters
        img = all_images[idx]  # (3, 64, 64)
        rgb = np.stack([img[2], img[1], img[0]], axis=-1)  # r, g, b order
        rgb = np.clip(rgb * 2.0, 0, 1)  # brighten for visibility

        ax.imshow(rgb, origin="lower")
        true_label = int(all_targets[idx])
        prob = all_probs[idx]
        pred_label = 1 if prob >= threshold else 0

        color = "green" if pred_label == true_label else "red"
        ax.set_title(
            f"True: {class_names[true_label]}\n"
            f"Pred: {class_names[pred_label]} ({prob:.3f})",
            fontsize=10, color=color, fontweight="bold",
        )
        ax.axis("off")

    axes[0, 0].set_ylabel("Lenses", fontsize=12, fontweight="bold")
    axes[1, 0].set_ylabel("Non-Lenses", fontsize=12, fontweight="bold")

    plt.suptitle("Sample Test Predictions", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "sample_predictions.png", bbox_inches="tight", dpi=150)
    plt.close()
    print("  📈 Saved sample_predictions.png")

    
def find_optimal_threshold(targets, probs):
    """Find threshold that maximizes F1 score (good for imbalanced data)."""
    best_f1, best_thresh = 0.0, 0.5
    for t in np.arange(0.05, 0.95, 0.01):
        preds = (probs >= t).astype(int)
        f1 = f1_score(targets, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
    return best_thresh, best_f1

In [32]:
def full_evaluation(model, test_loader, device, history_p1, history_p2):
    """Full evaluation on test set with all plots and metrics."""
    print("\n" + "=" * 70)
    print("📊 FINAL EVALUATION ON TEST SET")
    print("=" * 70)

    criterion = FocalLoss(alpha=0.75, gamma=2.0)
    test_loss, test_auc, all_targets, all_probs = evaluate(
        model, test_loader, criterion, device
    )

    # Find optimal threshold
    opt_thresh, opt_f1 = find_optimal_threshold(all_targets, all_probs)
    all_preds = (all_probs >= opt_thresh).astype(int)

    print(f"\n  Test AUC:          {test_auc:.4f}")
    print(f"  Optimal Threshold: {opt_thresh:.3f}")
    print(f"  F1 at Threshold:   {opt_f1:.4f}")

    # Classification report
    target_names = ["Non-Lens", "Lens"]
    report = classification_report(
        all_targets, all_preds, target_names=target_names, output_dict=True
    )
    print("\n  Classification Report:")
    print(classification_report(all_targets, all_preds, target_names=target_names))

    # === PLOT 1: Training Curves ===
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 120})

    epochs_p1 = range(1, len(history_p1["train_loss"]) + 1)

    # Loss
    axes[0].plot(epochs_p1, history_p1["train_loss"], "o-", label="Train Loss", markersize=3, color="#e74c3c")
    axes[0].plot(epochs_p1, history_p1["val_loss"], "s-", label="Val Loss", markersize=3, color="#3498db")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Focal Loss")
    axes[0].set_title("Phase 1: Training & Validation Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # AUC (Phase 1 + Phase 2)
    all_auc = history_p1["val_auc"] + history_p2["val_auc"]
    epochs_all = range(1, len(all_auc) + 1)
    p1_len = len(history_p1["val_auc"])
    axes[1].plot(epochs_all[:p1_len], all_auc[:p1_len], "o-", label="Phase 1 (Supervised)", markersize=3, color="#2ecc71")
    axes[1].plot(epochs_all[p1_len:], all_auc[p1_len:], "s-", label="Phase 2 (RL)", markersize=3, color="#9b59b6")
    axes[1].axvline(x=p1_len, color="gray", linestyle="--", alpha=0.5, label="Phase Boundary")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Validation AUC")
    axes[1].set_title("AUC Over Training (Both Phases)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # RL Reward
    epochs_p2 = range(1, len(history_p2["reward"]) + 1)
    ax2_twin = axes[2].twinx()
    ax2_twin.plot(epochs_p2, history_p2["threshold"], "s-", label="Avg Threshold", markersize=3, color="#1abc9c")
    axes[2].set_xlabel("Epoch")
    ax2_twin.set_ylabel("Threshold", color="#1abc9c")
    axes[2].set_title("Phase 2: RL Reward & Learned Threshold")
    lines1, labels1 = axes[2].get_legend_handles_labels()
    lines2, labels2 = ax2_twin.get_legend_handles_labels()
    axes[2].legend(lines1 + lines2, labels1 + labels2, loc="lower right")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_curves.png", bbox_inches="tight", dpi=150)
    plt.close()
    print("  📈 Saved training_curves.png")

    # === PLOT 2: ROC and Precision-Recall ===
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # ROC Curve
    fpr, tpr, _ = roc_curve(all_targets, all_probs)
    roc_auc_val = auc(fpr, tpr)
    ax1.plot(fpr, tpr, color="#e74c3c", linewidth=2.5, label=f"ROC (AUC = {roc_auc_val:.4f})")
    ax1.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random (AUC = 0.5)")
    ax1.fill_between(fpr, tpr, alpha=0.1, color="#e74c3c")
    ax1.set_xlabel("False Positive Rate")
    ax1.set_ylabel("True Positive Rate")
    ax1.set_title("ROC Curve")
    ax1.legend(loc="lower right", fontsize=11)
    ax1.grid(True, alpha=0.3)

    # Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(all_targets, all_probs)
    ap = average_precision_score(all_targets, all_probs)
    ax2.plot(recall, precision, color="#2ecc71", linewidth=2.5, label=f"PR (AP = {ap:.4f})")
    baseline_pr = all_targets.sum() / len(all_targets)
    ax2.axhline(y=baseline_pr, color="k", linestyle="--", alpha=0.4, label=f"Random (P = {baseline_pr:.4f})")
    ax2.fill_between(recall, precision, alpha=0.1, color="#2ecc71")
    ax2.set_xlabel("Recall")
    ax2.set_ylabel("Precision")
    ax2.set_title("Precision-Recall Curve")
    ax2.legend(loc="upper right", fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "roc_curve.png", bbox_inches="tight", dpi=150)
    plt.close()
    print("  📈 Saved roc_curve.png")

    # === PLOT 3: Confusion Matrix ===
    cm = confusion_matrix(all_targets, all_preds)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=target_names, yticklabels=target_names, ax=ax,
        annot_kws={"size": 14},
    )
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True", fontsize=12)
    ax.set_title(f"Confusion Matrix (threshold = {opt_thresh:.3f})", fontsize=13)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "confusion_matrix.png", bbox_inches="tight", dpi=150)
    plt.close()
    print("  📈 Saved confusion_matrix.png")

    # === PLOT 4: Sample Predictions ===
    plot_sample_predictions(model, test_loader, device, opt_thresh)

    # === Save Results ===
    results = {
        "test_auc": float(test_auc),
        "test_loss": float(test_loss),
        "optimal_threshold": float(opt_thresh),
        "f1_at_threshold": float(opt_f1),
        "average_precision": float(ap),
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
        "roc_data": {"fpr": fpr.tolist(), "tpr": tpr.tolist(), "auc": float(roc_auc_val)},
        "phase1_history": history_p1,
        "phase2_history": history_p2,
    }
    with open(OUTPUT_DIR / "results.json", "w") as f:
        json.dump(results, f, indent=2)
    print("  📁 Saved results.json")

    return results

In [38]:
set_seed()
print(f"🔭 Gravitational Lens Classifier")
print(f"   Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   Output: {OUTPUT_DIR}")

# Data
train_loader, val_loader, test_loader = create_dataloaders()
# Model

model = LensClassifier(dropout=0.3).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n🧠 Model: ResNet18-based LensClassifier")
print(f"   Total params:     {total_params:,}")
print(f"   Trainable params: {trainable_params:,}")

# Phase 1: Supervised
model, history_p1, p1_auc = train_phase1(model, train_loader, val_loader, DEVICE)
# Phase 2: RL Fine-tuning
model, policy, history_p2 = train_phase2_rl(model, train_loader, val_loader, DEVICE)


🔭 Gravitational Lens Classifier
   Device: cuda
   GPU: NVIDIA GeForce RTX 4060 Laptop GPU
   Output: /home/vidit68/Desktop/drive/Deeplense/specific_taskV
📊 Data Summary:
   Train: 25,845  (lens: 1,471, non-lens: 24,374)
   Val:   4,560  (lens: 259, non-lens: 4,301)
   Test:  19,650  (lens: 195, non-lens: 19,455)

🧠 Model: ResNet18-based LensClassifier
   Total params:     11,242,305
   Trainable params: 11,242,305

🚀 PHASE 1: Supervised Training (Focal Loss + Weighted Sampling)
  Epoch [01/30] | Train Loss: 0.0342 | Val Loss: 0.0505 | Val AUC: 0.9624 | LR: 0.000997 ✅ saved
  Epoch [02/30] | Train Loss: 0.0263 | Val Loss: 0.0181 | Val AUC: 0.9741 | LR: 0.000989 ✅ saved
  Epoch [03/30] | Train Loss: 0.0242 | Val Loss: 0.0385 | Val AUC: 0.9692 | LR: 0.000976
  Epoch [04/30] | Train Loss: 0.0222 | Val Loss: 0.0441 | Val AUC: 0.9778 | LR: 0.000957 ✅ saved
  Epoch [05/30] | Train Loss: 0.0209 | Val Loss: 0.0414 | Val AUC: 0.9773 | LR: 0.000933
  Epoch [06/30] | Train Loss: 0.0201 | Val Loss

In [40]:
# Full Evaluation
results = full_evaluation(model, test_loader, DEVICE, history_p1, history_p2)
print("\n" + "=" * 70)
print("✅ COMPLETE!")
print(f"   Phase 1 Best Val AUC: {p1_auc:.8f}")
print(f"   Phase 2 Best Val AUC: {max(history_p2['val_auc']):.4f}")
print(f"   Test AUC:             {results['test_auc']:.4f}")
print(f"   Test AP:              {results['average_precision']:.4f}")
print(f"   Optimal Threshold:    {results['optimal_threshold']:.3f}")
print("=" * 70)


📊 FINAL EVALUATION ON TEST SET

  Test AUC:          0.9868
  Optimal Threshold: 0.870
  F1 at Threshold:   0.6931

  Classification Report:
              precision    recall  f1-score   support

    Non-Lens       1.00      1.00      1.00     19455
        Lens       0.72      0.67      0.69       195

    accuracy                           0.99     19650
   macro avg       0.86      0.83      0.85     19650
weighted avg       0.99      0.99      0.99     19650

  📈 Saved training_curves.png
  📈 Saved roc_curve.png
  📈 Saved confusion_matrix.png
  📈 Saved sample_predictions.png
  📁 Saved results.json

✅ COMPLETE!
   Phase 1 Best Val AUC: 0.99076716
   Phase 2 Best Val AUC: 0.9908
   Test AUC:             0.9868
   Test AP:              0.7247
   Optimal Threshold:    0.870


# Training curves

![training-curve](./training_curves.png)


Since, the ROC is being optimized in the policy for RL i.e. phase-2 training. The fluctuations in the validation loss for phase-1 can be ignored.

# ROC and Precision recall curve

![roc](./roc_curve.png)

# Confusion matrix

![cfm](./confusion_matrix.png)


# Sample inference

![infer](sample_predictions.png)